# Biased RPS strategy trajectories

This notebook visualizes the strategy trajectories saved under `../nfg/logs/brps_trajectory/`.

**Run the trajectory-generation script first** (from the `nfg/` directory) so the CSV logs exist:

```bash
cd ../nfg
bash scripts/run_brps_trajectory.sh
```

- Cells below use `mu=1.5` (3 algorithms) and the `mu` sweep (`0.5, 1.0, 2.0, 4.0`).
- `run_brps_trajectory.sh` already generates both; running just `dump_brps_trajectory.py --mus 1.5 --algorithms gda symp_gda asymp_gda` is enough for the GIF cell alone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ternary

In [ ]:
LOG_ROOT = "../nfg/logs/brps_trajectory"
COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]
EQUILIBRIUM = [0.2, 0.6, 0.2]
INITIAL_STRATEGIES = [
    np.array([0.05, 0.9, 0.05]),
    np.array([0.1, 0.1, 0.8]),
    np.array([0.1, 0.3, 0.6]),
]


def load_trajectory(alg, mu, init_idx):
    path = f"{LOG_ROOT}/{alg}/mu{mu}/init{init_idx}.csv"
    df = pd.read_csv(path)
    return df[["x_0", "x_1", "x_2"]].to_numpy()


def setup_ternary(ax, scale=1.0):
    _, tax = ternary.figure(scale=scale, ax=ax)
    tax.boundary(linewidth=2.0)
    tax.gridlines(color="gray", multiple=0.1)
    tax.right_corner_label("$x_1$", fontsize=12)
    tax.top_corner_label("$x_2$", fontsize=12)
    tax.left_corner_label("$x_3$", fontsize=12)
    tax.get_axes().set_aspect(1)
    tax.ticks(axis="lbr", multiple=0.2, linewidth=1, tick_formats="%.1f")
    tax.get_axes().axis("off")
    tax.clear_matplotlib_ticks()
    for idx, x0 in enumerate(INITIAL_STRATEGIES):
        tax.scatter([x0], marker="o", color=COLORS[idx])
    tax.scatter([EQUILIBRIUM], marker="o", color="red", s=50, label="Equilibrium")
    return tax

In [ ]:
# Plot 1: 3 algorithms at mu=1.5.
algs = [("gda", "GDA"), ("symp_gda", "SymP-GDA"), ("asymp_gda", "AsymP-GDA")]
mu = 1.5

fig, axs = plt.subplots(1, len(algs), figsize=(13, 7))
for i_a, (alg, title) in enumerate(algs):
    tax = setup_ternary(axs[i_a])
    for i_init in range(len(INITIAL_STRATEGIES)):
        traj = load_trajectory(alg, mu, i_init)
        tax.plot(traj, linewidth=2.0, linestyle="-", markersize=3, color=COLORS[i_init])
    axs[i_a].set_title(title, fontsize=16, y=1.1)

plt.savefig("../nfg/logs/brps_x_ternary.pdf", bbox_inches="tight")

In [ ]:
# Animate the brps_x_ternary trajectories (mu=1.5) and export a GIF for the README.
import io

from PIL import Image, ImageChops

# (algorithm key, panel title, title color); AsymP-GDA is our method, highlighted in red.
algs = [
    ("gda", "GDA", "black"),
    ("symp_gda", "SymP-GDA", "black"),
    ("asymp_gda", "AsymP-GDA (Ours)", "#d62728"),
]
mu = 1.5
trajectories = {alg: [load_trajectory(alg, mu, i) for i in range(len(INITIAL_STRATEGIES))] for alg, _, _ in algs}
n_steps = trajectories["gda"][0].shape[0]

n_frames = 80
frame_ends = np.unique(np.linspace(1, n_steps, n_frames).astype(int))


def render(end):
    fig, axs = plt.subplots(1, len(algs), figsize=(13, 6))
    fig.subplots_adjust(top=0.86, bottom=0.0, left=0.01, right=0.99, wspace=0.05)
    for i_a, (alg, title, color) in enumerate(algs):
        tax = setup_ternary(axs[i_a])
        for i_init in range(len(INITIAL_STRATEGIES)):
            traj = trajectories[alg][i_init][:end]
            tax.plot(traj, linewidth=2.0, color=COLORS[i_init])
        axs[i_a].set_title(title, fontsize=24, y=1.02, color=color, fontweight="bold")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=80)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


frames = [render(end) for end in frame_ends]


def content_bbox(img):
    bg = Image.new("RGB", img.size, (255, 255, 255))
    return ImageChops.difference(img, bg).getbbox()


# Crop a common white border (removes the top/bottom margin) so the GIF is tight.
boxes = [content_bbox(f) for f in frames]
pad = 4
left = max(min(b[0] for b in boxes) - pad, 0)
upper = max(min(b[1] for b in boxes) - pad, 0)
right = min(max(b[2] for b in boxes) + pad, frames[0].width)
lower = min(max(b[3] for b in boxes) + pad, frames[0].height)
frames = [f.crop((left, upper, right, lower)) for f in frames]

size = frames[0].size
frames = [f if f.size == size else f.resize(size) for f in frames]

gif_path = "../assets/brps_x_ternary.gif"
frames.extend([frames[-1]] * 12)  # hold the final converged frame for ~1s
frames[0].save(gif_path, save_all=True, append_images=frames[1:], duration=83, loop=0)
print(f"saved {gif_path} ({len(frames)} frames, size={size})")

In [ ]:
# Preview the exported GIF inline.
from IPython.display import Image as IPyImage

IPyImage(filename=gif_path)

In [ ]:
# Plot 2: sym/asym across a mu sweep.
algs = [("symp_gda", "SymP-GDA"), ("asymp_gda", "AsymP-GDA")]
mus = [0.5, 1.0, 2.0, 4.0]

fig, axs = plt.subplots(len(algs), len(mus), figsize=(18, 8))
for i_m, mu in enumerate(mus):
    for i_a, (alg, title) in enumerate(algs):
        tax = setup_ternary(axs[i_a, i_m])
        for i_init in range(len(INITIAL_STRATEGIES)):
            traj = load_trajectory(alg, mu, i_init)
            tax.plot(traj, linewidth=2.0, linestyle="-", markersize=3, color=COLORS[i_init])
        axs[i_a, i_m].set_title(rf"{title} ($\mu={mu}$)", fontsize=16, y=1.1)

plt.savefig("../nfg/logs/brps_x_ternary_mus.pdf", bbox_inches="tight")